# Fine-tune the Fundamental Agent (LoRA) on Colab

Runs `finetune_lora.py` against `data/gold/fundamental_finetune_data.jsonl` using a 4-bit
(QLoRA) base model load, so it fits the free T4 GPU.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

**One-time setup (outside Colab):** upload `finetune_lora.py`, `infer_lora.py`, and
`fundamental_finetune_data.jsonl` to a folder in your Google Drive, e.g.
`My Drive/data298a/`. Everything below assumes that path.

## 1. Mount Drive and set the working folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/My Drive/data298a'
%cd {PROJECT_DIR}
!ls -la

## 2. Install dependencies

Colab already ships `torch` with CUDA; we only need the fine-tuning stack on top.

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes

## 3. Confirm the GPU is visible

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 4. Hugging Face login

`mistralai/Mistral-7B-Instruct-v0.3` is gated. Accept the license at
https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3 first, then log in
with a token from https://huggingface.co/settings/tokens (read access is enough).

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted

## 5. Run fine-tuning

4-bit QLoRA loading is automatic whenever a CUDA GPU is present (pass `--no-4bit`
to disable it). Output adapter is saved back into your Drive folder, so it survives
the Colab session ending.

In [ ]:
!python3 finetune_lora.py \
  --train-jsonl fundamental_finetune_data.jsonl \
  --base-model mistralai/Mistral-7B-Instruct-v0.3 \
  --output-dir fundamental_lora_adapter \
  --epochs 3 \
  --batch-size 1 \
  --grad-accum 8

## 6. Sanity-check the adapter with a quick inference call

The adapter directory (`fundamental_lora_adapter/`) is already in your Drive folder
— nothing further to download. Point `agents/infer_lora.py` at it from anywhere.

In [ ]:
!python3 infer_lora.py \
  --base-model mistralai/Mistral-7B-Instruct-v0.3 \
  --adapter-dir fundamental_lora_adapter \
  --prompt '{"ticker":"AAPL","features":{"debt_to_assets":0.25}}'